In [1]:
import numpy as np 
import matplotlib.pyplot as plt 

In [43]:
class Host:
    def __init__(self, x, y, threshold = 1, infect_rate = 0.5, total_n_pathogens = 30): # look at the standard values
        self.threshold = threshold                      # threshold above which host becomes diseased --> establishment
        self.infect_rate = infect_rate                  # rate of infection from this host species
        self.total_n_pathogens = total_n_pathogens      # total number of pathogens that the host can have: growth limit

        self.diseased = False                           # disease status of host
        self.n_pathogens = 0                            # number of pathogen particles in the host 

        self.x = x
        self.y = y

    def initial_diseased(self, n_pathogens):
        self.diseased = True
        self.n_pathogens += n_pathogens 

    def check_diseased(self):
        if self.n_pathogens > self.threshold:
            self.diseased = True

class Grid:
    def __init__(self, n_rows, n_cols, network_type, thresholds, infect_rates, total_n_pathogens):
        assert network_type in ["lattice_hex", "lattice_square", "random"], f"network_type needs to be one of: [lattice_hex, lattice_square, random], got {network_type}"
        
        self.n_rows = n_rows
        self.n_cols = n_cols
        self.network_type = network_type
        self.thresholds = thresholds                # establishment thresholds for all the hosts
        self.infect_rates = infect_rates            # infection rates for all the hosts (whether they are diseased or not)
        self.total_n_pathogens = total_n_pathogens  # pathogen growth limit for all the hosts

    def get_xy_stats(self, x_vals_grid, y_vals_grid):
        xy_stats = {}
        xy_stats["x_vals_grid"] = x_vals_grid; xy_stats["y_vals_grid"] = y_vals_grid
        x_vals = np.unique(x_vals_grid); y_vals = np.unique(y_vals_grid)

        xy_stats["xmin"] = np.min(x_vals_grid); xy_stats["xmax"] = np.max(x_vals_grid)
        xy_stats["ymin"] = np.min(y_vals_grid); xy_stats["ymax"] = np.max(y_vals_grid)
        xy_stats["dx"] = x_vals[1] - x_vals[0]; xy_stats["dy"] = y_vals[1] - y_vals[0]

        return xy_stats

class HexaGrid(Grid):
    def __init__(self, n_rows, n_cols, thresholds, infect_rates, total_n_pathogens):
        super().__init__(n_rows, n_cols, "lattice_hex", thresholds, infect_rates, total_n_pathogens)
        assert self.n_cols == self.n_rows * 2 - 1, f"Need a square grid to accurately compute n_hosts (for now): n_cols = n_rows * 2 - 1"
        self.n_hosts = self.n_cols * (self.n_rows // 2)

        assert len(self.thresholds) == self.n_hosts, f"Need to get enough threshold values for all hosts ({self.n_hosts}), got {len(self.thresholds)}"
        assert len(self.infect_rates) == self.n_hosts, f"Need to get enough infection rate values for all hosts ({self.n_hosts}), got {len(self.infect_rates)}"
        assert len(self.total_n_pathogens) == self.n_hosts, f"Need to get enough total n pathogen values for all hosts ({self.n_hosts}), got {len(self.total_n_pathogens)}"

    def get_neighbors(self, node, xy_stats):
        x, y = node
        dx = xy_stats["dx"]; dy = xy_stats["dy"]
        xmin = xy_stats["xmin"]; xmax = xy_stats["xmax"]
        ymin = xy_stats["ymin"]; ymax = xy_stats["ymax"]

        # Get hexagonal grid neighbors (for this data)
        neighbors = [(x + dx, y + dy), (x + dx, y - dy), (x + 2*dx, y), 
                    (x - dx, y + dy), (x - dx, y - dy), (x - 2*dx, y)]
        
        # Check whether the neighbors are within the limits of the grid
        valid_neighbors = []
        for x_nbr, y_nbr in neighbors:
            if xmin <= x_nbr <= xmax and ymin <= y_nbr <= ymax:
                valid_neighbors.append((x_nbr, y_nbr))
        
        return set(valid_neighbors)
        
    def build_lattice(self):
        x_vals = np.arange(0, self.n_cols, 1)
        y_vals = np.arange(0, self.n_rows, 1)

        x_vals_grid = []
        y_vals_grid = []

        self.node_dict = {}; node_idx = 0
        self.host_list = []
        for i, x in enumerate(x_vals):
            start = 0 if i % 2 == 0 else 1 # staggered columns in hexagonal grid
            for j in range(start, len(y_vals), 2):
                y = y_vals[j]

                threshold = self.thresholds[node_idx]
                infect_rate = self.infect_rates[node_idx]
                total_n_pathogen = self.total_n_pathogens[node_idx]

                self.host_list.append(Host(x, y, threshold, infect_rate, total_n_pathogen))
                self.node_dict[node_idx] = (x, y)

                x_vals_grid.append(x)
                y_vals_grid.append(y)
                node_idx += 1
        
        self.coord_to_idx = {coord: idx for idx, coord in self.node_dict.items()}
        self.xy_stats = super().get_xy_stats(np.array(x_vals_grid), np.array(y_vals_grid))

        neighbor_dict = {}; self.link_set = set()
        for node_ID, node in self.node_dict.items():
            node_nbrs = self.get_neighbors(node, self.xy_stats)
            nbr_IDs = [self.coord_to_idx[nbr] for nbr in node_nbrs]

            neighbor_dict[node_ID] = nbr_IDs

            for nbr_ID in nbr_IDs:
                link = tuple(sorted((node_ID, nbr_ID))) # make sure each link only gets added once
                self.link_set.add(link)

    def visualize(self, node_color = "forestgreen", link_color = "gray", show = True, lab = False):
        plt.figure(figsize = (6,6))
        for idx, node in self.node_dict.items():
            x, y = node
            if lab:
                lab = "Plant" if idx == 0 else ""
            else:
                lab = None

            plt.scatter(x, y, zorder = 2, color = node_color, s = 100, edgecolors = "white", 
                        marker = "s", label = lab)

        for link in self.link_set:
            node_ID1, node_ID2 = link
            coord1 = self.node_dict[node_ID1]; coord2 = self.node_dict[node_ID2]
            plt.plot([coord1[0], coord2[0]], [coord1[1], coord2[1]], zorder = 1, 
                     color = link_color, alpha = 0.5)
        plt.axis('off')
        if show:
            plt.show()

class DiseaseProgression:
    def __init__(self, n_hosts_infected, n_init_pathogens, grid, T_end, infection_type = "random"):
        assert infection_type in ["random"], f"infection_type needs to be in [random], got {infection_type}"
        self.infection_type = infection_type

        self.n_hosts_infected = n_hosts_infected
        self.n_init_pathogens = n_init_pathogens
        self.grid = grid
        self.T_end = T_end

    def initial_innoculation(self, seed = 3):
        np.random.seed(seed)
        if self.infection_type == "random":
            self.initial_diseased_hosts = np.random.choice(self.grid.host_list, size = self.n_hosts_infected, replace = False)

        for host in self.initial_diseased_hosts:
            host.initial_diseased(self.n_init_pathogens)

    def spread_disease(self):
        self.diseased_hosts = self.initial_diseased_hosts.copy()
        link_len = 1 # maybe change this later?

        t = 0
        times_until_infection = {} # list that contains duplicates: multiple hosts can have the same infect_rate
        for d_host in self.diseased_hosts:
            host_idx = self.grid.coord_to_idx[(d_host.x, d_host.y)]
            times_until_infection[host_idx] = t + d_host.infect_rate * link_len
    
        while t < self.T_end:
            t_next_infection = np.min(list(times_until_infection.values()))
            min_indices = [idx for idx in times_until_infection.keys() if times_until_infection[idx] == t_next_infection]
            print(times_until_infection)
            print(self.diseased_hosts)
            infecting_hosts = np.array(self.diseased_hosts)[min_indices]
            del times_until_infection[min_indices]
            
            new_diseased = set()
            for infecting_host in infecting_hosts:
                host_coords = infecting_host.x, infecting_host.y
                nbr_coords = self.grid.get_neighbors(host_coords, self.grid.xy_stats) # these are the coordinates, not the Host() objects or indices in the list

                for nbr_coord in nbr_coords:
                    nbr_idx = self.grid.coord_to_idx[nbr_coord]
                    nbr = self.grid.host_list[nbr_idx] # the indices of coord_to_idx match the indices in host_list by design: see HG.build_lattice

                    nbr.n_pathogens += 1
                    nbr.check_diseased()

                    if nbr.diseased:
                        if nbr not in self.diseased_hosts:
                            self.diseased_hosts.sppend(nbr)
                        new_diseased.add(nbr)

            t = t_next_infection 
            new_diseased -= set(self.diseased_hosts) # avoid including hosts that have become diseased already
            for new_host in new_diseased:
                new_host_idx = self.grid.coord_to_idx[(new_host.x, new_host.y)]
                times_until_infection[new_host_idx] = t + new_host.infect_rate * link_len

            if len (new_diseased) > 0:
                self.visualize_disease(show = True)

    def visualize_disease(self, show = True, lab = False, show_pathogen = False):
        # use Polar coordinates to rescale the speed of the pathogen particles
        def cart_to_polar(x, y):
            r = np.sqrt(x ** 2 + y ** 2)
            phi = np.arccos(x / r) if y >= 0 else -np.arccos(x / r)
            return r, phi

        def polar_to_cart(r, phi):
            x = r * np.cos(phi)
            y = r * np.sin(phi)
            return x, y 
        
        self.grid.visualize(show = False, lab = lab)
        for i, init_diseased_host in enumerate(self.initial_diseased_hosts):
            plt.scatter(init_diseased_host.x, init_diseased_host.y, color = "red", s = 180, 
                        zorder = 1, marker = "s")
        
        if show_pathogen:
            idx = 0
            for diseased_host in self.initial_diseased_hosts:
                host_x, host_y = diseased_host.x, diseased_host.y
                infect_rate = diseased_host.infect_rate
                nbr_coords = self.grid.get_neighbors((host_x, host_y), self.grid.xy_stats)
                for nbr in nbr_coords:
                    x1, y1 = nbr[0], nbr[1]
                    x2, y2 = host_x, host_y
                    
                    line_segment = np.array([x2 - x1, y2 - y1])
                    r, phi = cart_to_polar(line_segment[0], line_segment[1])
                    r_inf = r * (1- infect_rate)
                    x_inf, y_inf = polar_to_cart(r_inf, phi)

                    plt.scatter(x_inf + x1, y_inf+ y1, color = "red", s = 30, zorder = 4, 
                                label = "Pathogen" if idx == 0 else "")
                    idx += 1
            plt.legend(fontsize = 15, bbox_to_anchor = [1.5, 1.01], loc = "upper right", frameon = False)

        if show:
            plt.show()


In [44]:
n_rows = 6; n_cols = n_rows * 2 - 1 # to make a square
total_n_hosts = n_cols * (n_rows // 2)

thresholds = np.ones(total_n_hosts)
infect_rates = np.ones(total_n_hosts)
total_n_pathogens = np.ones(total_n_hosts) * 80

HG = HexaGrid(n_rows, n_cols, thresholds, infect_rates, total_n_pathogens)
HG.build_lattice()

DP = DiseaseProgression(HG.n_hosts // 10, 10, HG, 100)
DP = DiseaseProgression(HG.n_hosts // 10, 80, HG, T_end = 10)
DP.initial_innoculation()
DP.spread_disease()

{4: np.float64(1.0), 17: np.float64(1.0), 15: np.float64(1.0)}


IndexError: index 4 is out of bounds for axis 0 with size 3

In [38]:
time_list = [2,3,4,1,1,5]; times_until_infection = {}
for i in range(6):
    times_until_infection[i] = time_list[i]

diseased_hosts = ["a", "b", "c", "d", "e", "f"]

t_next_infection = np.min(list(times_until_infection.values()))
min_indices = [idx for idx in times_until_infection.keys() if times_until_infection[idx] == t_next_infection]

infecting_hosts = np.array(diseased_hosts)[min_indices]
infecting_hosts

array(['d', 'e'], dtype='<U1')